### Cleaning and preprocessing for later modeling

This note book prepares the fraud dataset for modeling by:

-shrink the scope to relevant transaction types

-create a few useful features

-sampling the data so the modeling is practical

-split the data into train/test sets

### Import Libraries

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


### Load the raw dataset

In [4]:
df = pd.read_csv("../data/raw/PS_20174392719_1491204439457_log.csv")
df.head()


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


### Inspect shape, columns and class balance

In [5]:
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nClass balance:\n", df["isFraud"].value_counts())
print("\nClass proportion:\n", df["isFraud"].value_counts(normalize=True))


Shape: (6362620, 11)

Columns:
 ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

Class balance:
 isFraud
0    6354407
1       8213
Name: count, dtype: int64

Class proportion:
 isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64


### Check missing values and duplicates

In [6]:
print("Missing values:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())


Missing values:
 step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

Duplicate rows: 0


### Modeling Scope

-Fraud only appears in TRANSFER and CASH_OUT

-To keep the project computationally reasonable and focused, the modeling data set will only use those transaction types

### Filter to relevant transactions types

In [7]:
df = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()
print(df.shape)
print(df["type"].value_counts())
print(df["isFraud"].value_counts())


(2770409, 11)
type
CASH_OUT    2237500
TRANSFER     532909
Name: count, dtype: int64
isFraud
0    2762196
1       8213
Name: count, dtype: int64


### Balance-error features

In [8]:
df["orig_balance_error"] = df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
df["dest_balance_error"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]

df["orig_error_flag"] = (df["orig_balance_error"].abs() > 1).astype(int)
df["dest_error_flag"] = (df["dest_balance_error"].abs() > 1).astype(int)


### Transaction type features


In [9]:
df["is_transfer"] = (df["type"] == "TRANSFER").astype(int)
df["is_cash_out"] = (df["type"] == "CASH_OUT").astype(int)


### Transformed amount feature

In [10]:
df["log_amount"] = np.log1p(df["amount"])


### Dropping columns I don't want to model directly


In [11]:
df = df.drop(columns=["nameOrig", "nameDest", "type"])
df.head()


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,orig_balance_error,dest_balance_error,orig_error_flag,dest_error_flag,is_transfer,is_cash_out,log_amount
2,1,181.00,181.0,0.0,0.0,0.00,1,0,0.00,181.0,0,1,1,0,5.204007
3,1,181.00,181.0,0.0,21182.0,0.00,1,0,0.00,21363.0,0,1,0,1,5.204007
15,1,229133.94,15325.0,0.0,5083.0,51513.44,0,0,-213808.94,182703.5,1,1,0,1,12.342066
19,1,215310.30,705.0,0.0,22425.0,0.00,0,0,-214605.30,237735.3,1,1,1,0,12.279840
24,1,311685.89,10835.0,0.0,6267.0,2719172.89,0,0,-300850.89,-2401220.0,1,1,1,0,12.649754


### Reasoning

-The full dataset is huge

-To make training practical the modeling dataset will include all fraud cases and a random sample of non-fraud cases.

-The aim is to preserve fraud examples while reducing runtime

### Smaller modeling dataset

In [12]:
fraud_df = df[df["isFraud"] == 1]
nonfraud_df = df[df["isFraud"] == 0].sample(n=50000, random_state=42)

model_df = pd.concat([fraud_df, nonfraud_df], axis=0)
model_df = model_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(model_df.shape)
print(model_df["isFraud"].value_counts())
print(model_df["isFraud"].value_counts(normalize=True))


(58213, 15)
isFraud
0    50000
1     8213
Name: count, dtype: int64
isFraud
0    0.858915
1    0.141085
Name: proportion, dtype: float64


### Seperate features and target

In [13]:
X = model_df.drop(columns="isFraud")
y = model_df["isFraud"]


### Train/test split

In [14]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)


### Check split sizes

In [15]:
print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)

print("\nTrain fraud rate:", y_train.mean())
print("Validation fraud rate:", y_val.mean())
print("Test fraud rate:", y_test.mean())


Train shape: (40749, 14) (40749,)
Validation shape: (8732, 14) (8732,)
Test shape: (8732, 14) (8732,)

Train fraud rate: 0.14108321676605562
Validation fraud rate: 0.14109024278515803
Test fraud rate: 0.14109024278515803


### Save processed datasets

In [16]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_val.to_csv("../data/processed/X_val.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_val.to_csv("../data/processed/y_val.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)


### Reasoning and conclusion

-data was filtered, cleaned and feature engineered

-smaller modeling dataset was created

-train/test sets were saved for later modeling in modeling.ipynb